Initialize Instruction Memory, Data Memory, and Program

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#Simulation perameters
Max_steps = 10000
imem_size = 256
DMEM_size = 256
Num_reg = 16

signal_size = 1000

#Mem_port is used for structural hazards (MAY NOT USE)
mem_port = 3
pc = 0

#Initializing opcodes for easy access
oplist = ['nop','load','store','BEQ','MAC','Load_smp','Shift_buff','jmp','halt']
op = {}

for i in range(len(oplist)):
    op[oplist[i]] = i

#Initializing register names
reg = {}
for i in range(Num_reg):
    reg['R'+str(i)] = i

#Defining status dict
status = { "Running": 0,
           "Stalled": 1,
           "Done": -1
          }

#Defining stages for simuation
#Fetch, Decode, Execute,Memory Opperation, Writeback
stages = ["IF","ID","EX","MEM","WB"]

stage = {}
for i in range(len(stages)):
    stage[stages[i]] = i

dmem = np.zeros((DMEM_size,),dtype=np.int32)


Generate signals

In [ ]:
t = np.linspace(0, signal_size, num=signal_size)
s_norm = np.random.randn(signal_size)

n_comp = 5

#Generating random signal components
amps = np.random.uniform(0.01,20,n_comp)
freqs = np.random.uniform(0.1,20,n_comp)
phases = np.random.uniform(0,2*np.pi,n_comp)

for i in range(n_comp):
    s_norm += amps[i]*np.sin(2*np.pi*freqs[i]*t + phases[i])

#Normalze signal
s_norm = s_norm/np.max(s_norm)

Creating Instruction and Pipeline Infrastructures

In [ ]:
instruction_dtype = np.dtype([ #Assuming instructions follow: rd = opcode r1 r2
    ("disp","U40"),
    ("opcode",np.int32),
    ("rd",np.int32),
    ("r1",np.int32),
    ("r2",np.int32),
    ("im",np.int32),          #For other cases (i.e jump, branch, compare), im is used to store immediate values too
    ("status",np.int32),
])

imem = np.zeros(imem_size,dtype=instruction_dtype)
branch = 0
pipeline = np.zeros(5,dtype=instruction_dtype)

def Execute(inst,dmem):
  op = inst["opcode"]
  rd = inst["rd"]
  r1 = inst["r1"]
  r2 = inst["r2"]
  return 0 #TODO

def Check_Hazards(pipeline): #Only data and control hazards. Used AFTER Execute Function to count hazards, and stall when needed
  hazards = []
#Handling RAW Data hazard
  if (pipeline[stage['WB']]['rd'] in (pipeline[stage['EX']]['r1'], pipeline[stage['EX']]['r2'], pipeline[stage['ID']]['r1'], pipeline[stage['ID']]['r2'])):
    pipeline["status"] = status["Stalled"]
    pipeline[stage['WB']]["status"] = status['Running']
    hazards.append("RAW")

  #For now I'm assuming theres no instructions being executed out of order
  #If we do use an archetecture that does, we'll need to add cases for WAW and WAR

  #Checking for Control Hazards
  if pipeline[stage['EX']]["opcode"] == op['BEQ']:

    if branch: #If a branch occurs, flush the pipeline by replacing the instructions behind the branch with nop. CHANGE ME IF NEEDED!
      hazards.append("Control")
      pipeline[stage['IF']:stage['EX']]["opcode"] = op['nop']
      pipeline[stage['IF']:stage['EX']]["status"] = status['Done']


  #If there is a control hazard, the pipeline is flushed. This means data hazards are avoided as the nothing would be reading after writing until the next cycle
  if "Control" in hazards:
    return "Control"

  elif "RAW" in hazards:
    return "RAW"

  else:
    return "None"




Circular Buffer Code (Using OOP for multiple buffers if needed)

In [ ]:
class circular_buffer:
  def __init__(self,size):
    self.size = size
    self.head = 0
    self.count = 0
    self.buffer = np.zeros(size)

  def push(self,x):
    self.buffer[self.head] = x
    self.head = (self.head + 1)%self.size
    return 1

#Tail pointer not needed as our continuous stream only needs to cycle data, not remove it. No pop functionality needed either

buff1 = circular_buffer(10)



Main Simulation
PROGRAM
   1. Update circular buffer with new meassurement (Load_smp)
   2. Preform MAC for 1st FIR (Could be done via vectorization)
   3. Second round of FIR
   4. Either check for Earthquake Frequency (TTF) or Use STA/LTA (Check once researched more)
   5. If threshold value found, trigger earthquake warning flag (Probably using a register to announce it)
   6. Go to (1) until data is no longer received (Or maybe hault when Earthquake detected)
   
   Assigned Registers:
   - R1: pointer to circular buffer
   - R2: pointer to weights for FIR 1
   - R3: pointer to weights for FIR 2
   - R4: Accumulate reg for FIR 1
   - R5: Accumulate reg for FIR 2
   - R6: Flag register (Mostly to trigger Earthquake detection)
   
   - Other registers may be needed for pointing to memory or holding temporary results